# Bước 2: Xác minh Labels → YOLO Format

BDD100K labels có format box2d (absolute coordinates):
```
{"x1": 1125.9, "y1": 133.2, "x2": 1157.0, "y2": 210.9}
```

YOLO cần format YOLO (normalized xywh):
```
<class> <x_center> <y_center> <width> <height>
```

Notebook này kiểm tra:
1. Tất cả classes trong dataset
2. Convert box2d → YOLO format
3. Tạo labels cho một sample

In [ ]:
import json
import os
from collections import Counter
from pathlib import Path

## 1. Load metadata

In [ ]:
# Đường dẫn đã xác minh
METADATA_PATH = "/kaggle/input/datasets/solesensei/solesensei_bdd100k/bdd100k_labels_release/bdd100k/labels/bdd100k_labels_images_train.json"
IMAGE_DIR = "/kaggle/input/datasets/solesensei/solesensei_bdd100k/bdd100k/bdd100k/images/100k/train/trainA"

print("Loading metadata...")
with open(METADATA_PATH, 'r') as f:
    metadata = json.load(f)

print(f"Total records: {len(metadata)}")
print(f"\nSample record keys: {list(metadata[0].keys())}")

## 2. Xác minh Classes

In [ ]:
# Thu thập tất cả các category
all_categories = []

for record in metadata[:1000]:  # Check 1000 records
    for label in record.get('labels', []):
        category = label.get('category', 'UNKNOWN')
        all_categories.append(category)

print("All categories found:")
for cat, count in Counter(all_categories).most_common():
    print(f"  {cat}: {count}")

## 3. BDD100K → YOLO Class Mapping

In [ ]:
# BDD100K categories → YOLO class indices
# Theo config trong trainer.py

BDD100K_TO_YOLO = {
    'pedestrian': 0,
    'rider': 1,
    'other person': 2,
    'bicycle': 3,
    'car': 4,
    'bus': 5,
    'truck': 6,
    'motorcycle': 7,
    'traffic light': 8,
    'traffic sign': 9,
}

print("BDD100K → YOLO Class Mapping:")
for bdd_cat, yolo_idx in BDD100K_TO_YOLO.items():
    print(f"  {yolo_idx}: {bdd_cat}")

## 4. Convert box2d → YOLO format

In [ ]:
def convert_box2d_to_yolo(box2d, img_width, img_height):
    """
    Convert BDD100K box2d (absolute) → YOLO format (normalized xywh).
    
    box2d: {"x1": float, "y1": float, "x2": float, "y2": float}
    img_width: Chiều rộng ảnh (pixel)
    img_height: Chiều cao ảnh (pixel)
    
    Returns: (class_id, x_center_norm, y_center_norm, width_norm, height_norm)
    """
    
    # Calculate box dimensions
    x1, y1 = box2d['x1'], box2d['y1']
    x2, y2 = box2d['x2'], box2d['y2']
    
    # Width và height
    box_width = x2 - x1
    box_height = y2 - y1
    
    # Center
    x_center = x1 + box_width / 2
    y_center = y1 + box_height / 2
    
    # Normalize
    x_center_norm = x_center / img_width
    y_center_norm = y_center / img_height
    width_norm = box_width / img_width
    height_norm = box_height / img_height
    
    return x_center_norm, y_center_norm, width_norm, height_norm


# Test với sample
sample_record = metadata[0]
print(f"Filename: {sample_record['name']}")
print(f"Size: {sample_record.get('attributes', {}).get('size', 'N/A')}")

# Lấy sample label
if sample_record.get('labels'):
    sample_label = sample_record['labels'][0]
    print(f"\nSample label: {sample_label['category']}")
    print(f"box2d: {sample_label['box2d']}")
    
    # Giả sử ảnh có kích thước 1280x720 (BDD100K default)
    img_w, img_h = 1280, 720
    
    x_c, y_c, w, h = convert_box2d_to_yolo(sample_label['box2d'], img_w, img_h)
    class_id = BDD100K_TO_YOLO.get(sample_label['category'], -1)
    
    print(f"\nYOLO format: {class_id} {x_c:.6f} {y_c:.6f} {w:.6f} {h:.6f}")

## 5. Kiểm tra Image Size

In [ ]:
# BDD100K images có kích thước cố định
# Kiểm tra attributes có chứa size không

sizes = []
for record in metadata[:100]:
    size = record.get('attributes', {}).get('size')
    if size:
        sizes.append(tuple(size))

print(f"Records with size info: {len(sizes)}/100")
if sizes:
    print(f"Unique sizes: {set(sizes)}")
else:
    print("\nNo size info in attributes.")
    print("BDD100K images typically are 1280x720.")
    print("Will assume: width=1280, height=720")

## 6. Tạo Labels cho Sample

In [ ]:
# Tạo YOLO labels cho record đầu tiên
sample_record = metadata[0]

# BDD100K default size
IMG_WIDTH = 1280
IMG_HEIGHT = 720

print(f"Creating YOLO labels for: {sample_record['name']}")
print(f"Image size: {IMG_WIDTH}x{IMG_HEIGHT}")
print(f"Number of objects: {len(sample_record.get('labels', []))}")
print("\nYOLO format labels:")
print("=" * 50)

yolo_labels = []
for label in sample_record.get('labels', []):
    category = label['category']
    box2d = label['box2d']
    
    if category not in BDD100K_TO_YOLO:
        print(f"  ⚠️ Unknown category: {category}")
        continue
    
    class_id = BDD100K_TO_YOLO[category]
    x_c, y_c, w, h = convert_box2d_to_yolo(box2d, IMG_WIDTH, IMG_HEIGHT)
    
    yolo_line = f"{class_id} {x_c:.6f} {y_c:.6f} {w:.6f} {h:.6f}"
    yolo_labels.append(yolo_line)
    print(f"{category:15} → {yolo_line}")

print("\n" + "=" * 50)
print("\nFull YOLO label file content:")
print("\n".join(yolo_labels))

## 7. Xác minh Labels tồn tại trên Disk

In [ ]:
# Kiểm tra xem labels đã được convert sẵn chưa
# Thử tìm .txt files cùng thư mục với ảnh

sample_filename = sample_record['name']
stem = Path(sample_filename).stem

# Thử các đường dẫn khác nhau
possible_label_paths = [
    f"{IMAGE_DIR}/../labels/train/{stem}.txt",
    f"{IMAGE_DIR}/labels/{stem}.txt",
    f"{IMAGE_DIR}/{stem}.txt",
]

print(f"Checking for existing labels: {stem}.txt")
for p in possible_label_paths:
    exists = os.path.exists(p)
    status = "✅" if exists else "❌"
    print(f"  {status} {p}")

## 8. Tổng kết

In [ ]:
print("=" * 60)
print("KẾT QUẢ XÁC MINH LABELS")
print("=" * 60)

print("""
✅ Labels có trong metadata với box2d coordinates
✅ Có đủ 10 classes theo BDD100K
✅ Convert được box2d → YOLO format

⚠️ CẦN THÊM: Hàm convert labels khi chuẩn bị dataset
   - Labels cần được convert từ metadata sang YOLO .txt format
   - Hoặc cần một bước preprocessing để tạo labels

Đường dẫn sử dụng:
METADATA_PATH = "/kaggle/input/datasets/solesensei/solesensei_bdd100k/bdd100k_labels_release/bdd100k/labels/bdd100k_labels_images_train.json"
IMAGE_DIR = "/kaggle/input/datasets/solesensei/solesensei_bdd100k/bdd100k/bdd100k/images/100k/train/trainA"
""")

print("=" * 60)